<a href="https://colab.research.google.com/github/tejashwinirk/Agentic-AI/blob/main/lab4_AgenticAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-classic langchain-community langchain-groq langchain-huggingface chromadb sentence-transformers pypdf -q
from google.colab import userdata
import os

# Installation including the modern huggingface integration package
print('Installation complete. Remember to set your GROQ_KEY in the Secrets tab!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.2/382.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204

In [2]:
from google.colab import files
uploaded = files.upload()          # click and choose a PDF
pdf_name = list(uploaded.keys())[0]
print('Uploaded:', pdf_name)

Saving no personal details.pdf to no personal details.pdf
Uploaded: no personal details.pdf


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
pages = PyPDFLoader(pdf_name).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print('Number of chunks:', len(chunks))

/tmp/ipykernel_5141/1009507550.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of chunks: 6


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
print("Downloading and initializing HuggingFaceEmbeddings model...")
from langchain_huggingface import HuggingFaceEmbeddings

# Using the modern langchain-huggingface integration
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
print("HuggingFaceEmbeddings model initialized!")

# Initializing the vector store with processed chunks
db = Chroma.from_documents(chunks, embeddings)
print('Vector database ready!')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFaceEmbeddings model initialized!
Vector database ready!


In [5]:
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata
import os

try:
    groq_api_key = userdata.get('Agentic')
except userdata.SecretNotFoundError:
    raise ValueError("GROQ_KEY not found in Secrets. Please add it in the left sidebar (key icon).")

# 1. Initialize your LLM using the secret key
llm = ChatGroq(api_key=groq_api_key, model='llama-3.1-8b-instant', temperature=0)

# 2. Define your prompt structure
system_prompt = (
    'Use the following pieces of retrieved context to answer the question. '
    'If you don\'t know the answer, say that you don\'t know.\n\n'
    'Context:\n{context}'
)
prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '{input}'),
])
# 3. Create the modern retrieval chain via langchain_classic
question_answer_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(db.as_retriever(), question_answer_chain)

# 4. Invoke the chain
query = 'What is this document about?'
response = qa_chain.invoke({'input': query})

print(response['answer'])

This document appears to be a resume or a professional profile for a person named Tejashwini R K, who is an aspiring data analyst. It highlights her educational background, professional summary, skills, and certifications in data analytics and related fields.
